# 🧬 AlphaFold3 Structure Prediction Example

This notebook demonstrates how to predict protein and protein-ligand complex structures using **AlphaFold3**.

## 📖 What is AlphaFold3?

AlphaFold3 is the latest version of Google DeepMind's revolutionary structure prediction model. It extends beyond proteins to predict structures of nucleic acids, ligands, and their complexes with state-of-the-art accuracy.

### Key Features:
- **State-of-the-art accuracy** -- Benchmark-leading performance
- **Multi-modal** -- Supports proteins, DNA, RNA, and ligands
- **Complex prediction** -- Handles protein-ligand, protein-DNA/RNA, and multi-chain complexes
- **Confidence scores** -- Provides pLDDT, pTM, ipTM, and composite ranking score
- **MSA-based** -- Uses multiple sequence alignments for evolutionary context
- **Diffusion sampling** -- Multiple samples per seed for robust predictions
- **Quality control** -- Built-in clash detection and disorder assessment

## 📥 Imports

## 📦 Shared Data Types

### `StructurePredictionComplex`
A biological complex containing one or more molecular chains to predict together.

| Field | Type | Description |
|-------|------|-------------|
| `chains` | `List[Chain]` | Chains in the complex. Accepts strings, `Chain` objects, or dicts |

### `Chain`
A single molecular chain with optional modifications.

| Field | Type | Description |
|-------|------|-------------|
| `sequence` | `str` | Sequence (protein amino acids, DNA/RNA bases, or ligand SMILES) |
| `entity_type` | `Optional[str]` | `"protein"`, `"dna"`, `"rna"`, or `"ligand"`. Auto-inferred if `None` |
| `modifications` | `List[ChainModification]` | Post-translational or nucleotide modifications. Default: `[]` |

### `ChainModification`
A modification at a specific position in a chain, specified using CCD codes.

| Field | Type | Description |
|-------|------|-------------|
| `position` | `int` | 1-based position in the sequence |
| `modification_code` | `str` | CCD code (e.g., `"SEP"` for phosphoserine, `"TPO"` for phosphothreonine) |

### `Structure`
A predicted 3D structure with coordinates, confidence metrics, and export methods.

## 📥 Imports

In [1]:
from pathlib import Path

from proto_tools import (
    AlphaFold3Config,
    AlphaFold3Input,
    Chain,
    StructurePredictionComplex,
    run_alphafold3,
)

## 🔍 Protein-Ligand Complex Prediction (MfnG Example)

Let's predict the structure of MfnG protein bound to L-tyrosine ligand.

### API Reference

**`AlphaFold3Input`**

| Field | Type | Description |
|-------|------|-------------|
| `complexes` | `List[StructurePredictionComplex]` | List of complexes to predict structures for |

**`AlphaFold3Config`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `name` | `str` | `"af3_job"` | Name of the AlphaFold3 folding job |
| `seeds` | `List[int]` | `[0]` | Seeds to use for AlphaFold3 (5 diffusion samples per seed) |
| `use_msa` | `bool` | `True` | Whether to generate and use MSAs for protein chains |

**`AlphaFold3Output`**

| Field | Type | Description |
|-------|------|-------------|
| `structures` | `List[Structure]` | List of predicted structures, one per input complex |

**Note:** AlphaFold3 automatically converts SMILES strings to CCD (Chemical Component Dictionary) codes for ligands.

In [2]:
mfng_sequence = "MVTPEGNVSLVDESLLVGVTDEDRAVRSAHQFYERLIGLWAPAVMEAAHELGVFAALAEAPADSGELARRLDCDARAMRVLLDALYAYDVIDRIHDTNGFRYLLSAEARECLLPGTLFSLVGKFMHDINVAWPAWRNLAEVVRHGARDTSGAESPNGIAQEDYESLVGGINFWAPPIVTTLSRKLRASGRSGDATASVLDVGCGTGLYSQLLLREFPRWTATGLDVERIATLANAQALRLGVEERFATRAGDFWRGGWGTGYDLVLFANIFHLQTPASAVRLMRHAAACLAPDGLVAVVDQIVDADREPKTPQDRFALLFAASMTNTGGGDAYTFQEYEEWFTAAGLQRIETLDTPMHRILLARRATEPSAVPEGQASENLYFQ"

# L-tyrosine SMILES
tyrosine_smiles = "c1cc(ccc1C[C@@H](C(=O)O)N)O"

# Create protein-ligand complex
complex = StructurePredictionComplex(
    chains=[
        Chain(sequence=mfng_sequence, entity_type="protein"),
        Chain(sequence=tyrosine_smiles, entity_type="ligand"),
    ]
)

# Create input
inputs = AlphaFold3Input(complexes=[complex])

# Configure AlphaFold3
config = AlphaFold3Config(
    name="mfng_ligand",
    seeds=[0],  # AlphaFold3 generates 5 diffusion samples per seed
    use_msa=False,
    verbose=False,
)

# Run prediction
result = run_alphafold3(inputs, config)
mfng_structure = result.structures[0]

Folding structures (AlphaFold3): 100%|██████████| 1/1 [01:08<00:00, 68.95s/complex]


In [3]:
# Print metrics
print(f"\n✅ Structure predicted!")
print(f"  Number of chains: {len(complex.chains)}")
print(f"  Protein length: {len(mfng_sequence)} residues")
print(f"  Ranking score: {mfng_structure.ranking_score:.3f}")
print(f"  Average pLDDT: {mfng_structure.avg_plddt:.3f}")
print(f"  pTM score: {mfng_structure.ptm:.3f}")
print(f"  ipTM score: {mfng_structure.iptm:.3f}")


✅ Structure predicted!
  Number of chains: 2
  Protein length: 384 residues
  Ranking score: 0.930
  Average pLDDT: 75.450
  pTM score: 0.700
  ipTM score: 0.920


### 🎨 Visualize MfnG-Ligand Complex

In [4]:
mfng_structure.visualize(style='stick', color_by='chain')

## 💾 Export Results

Save predicted structures for further analysis in other tools like PyMOL, ChimeraX, or VMD.

### Supported formats:
- **PDB** -- Standard protein data bank format, widely compatible
- **mmCIF** -- Modern crystallographic information file, supports larger structures

The B-factor column contains the pLDDT scores for confidence visualization.

In [ ]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Export results to pdb files
result.export(name="mfng_ligand_complex", export_path=output_dir, file_format="pdb")